In [ ]:
# ============================================================
# GOOGLE COLAB VERSION
# AI-Assisted Hybrid Cryptosystem
# Permutation Matrix (from Bipartite Graph Kp,q) + AES-CBC
# ============================================================

!pip install pycryptodome

import numpy as np
import os
import time
import hashlib

from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from google.colab import files

# ============================================================
# ENTROPY CALCULATION
# ============================================================

def hitung_entropy(data):
    hist = np.bincount(data, minlength=256)
    prob = hist / np.sum(hist)
    entropy = -np.sum([p * np.log2(p) for p in prob if p > 0])
    return entropy

# ============================================================
# AI MATRIX SIZE SELECTOR
# ============================================================

def pilih_matriks_ai(data):
    filesize = len(data)
    entropy = hitung_entropy(data)

    if filesize < 50000:
        n = 4 if entropy < 5 else 6
    elif filesize < 500000:
        n = 8 if entropy < 6 else 12
    elif filesize < 5000000:
        n = 16 if entropy < 7 else 24
    else:
        n = 32 if entropy < 7.5 else 48

    print(f"\nAI Selected Matrix Size : {n}")
    return n

# ============================================================
# PERMUTATION MATRIX GENERATOR (from Kp,q)
# ============================================================

def kunci_permutasi(p, q, ed='e'):
    n = p + q
    P = np.eye(n, dtype=np.int32)

    # contoh permutasi sederhana: tukar beberapa simpul di partisi V
    perm = list(range(n))
    if q >= 2:
        perm[p], perm[p+1] = perm[p+1], perm[p]
    if q >= 4:
        perm[p+2], perm[p+3] = perm[p+3], perm[p+2]

    P = P[perm, :]

    if ed == 'e':
        return P, 0
    elif ed == 'd':
        return P, P.T   # invers = transpose

# ============================================================
# AES ENCRYPTION
# ============================================================

def aes_encrypt(pesan, kunci):
    data = bytes(pesan)
    key = hashlib.sha256(kunci.encode()).digest()
    cipher = AES.new(key, AES.MODE_CBC)
    iv = cipher.iv
    padded_data = pad(data, AES.block_size)
    ciphertext = cipher.encrypt(padded_data)
    return list(iv + ciphertext)

# ============================================================
# AES DECRYPTION
# ============================================================

def aes_decrypt(data, kunci):
    key = hashlib.sha256(kunci.encode()).digest()
    data = bytes(data)
    iv = data[:AES.block_size]
    ciphertext = data[AES.block_size:]
    cipher = AES.new(key, AES.MODE_CBC, iv=iv)
    decrypted_padded = cipher.decrypt(ciphertext)
    decrypted = unpad(decrypted_padded, AES.block_size)
    return list(decrypted)

# ============================================================
# MD5 HASH
# ============================================================

def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# ============================================================
# ENCRYPTION
# ============================================================

def enkripsi(fileasli, password2):
    mp = np.fromfile(fileasli, dtype=np.uint8)
    n = pilih_matriks_ai(mp)

    # tentukan p,q (misalnya p = n//2, q = n - p)
    p, q = n//2, n - n//2

    x0_str = "0." + str(password2) + "1"

    len_shift = mp.size % n
    len_hill = mp.size - len_shift

    kunciku, _ = kunci_permutasi(p, q, 'e')

    mp_reshape = mp[:len_hill].reshape((n, int(len_hill / n)))
    m_kali = np.dot(kunciku.astype(np.int32), mp_reshape.astype(np.int32)) % 256
    m_kali = m_kali.astype(np.uint8)

    maes = aes_encrypt(mp[len_hill:], x0_str)

    m_kali_1d = m_kali.reshape(len_hill)
    header_n = np.array(list(n.to_bytes(4, byteorder='big')), dtype=np.uint8)
    header_len = np.array(list(len_hill.to_bytes(8, byteorder='big')), dtype=np.uint8)

    m_cipher = np.concatenate((header_n, header_len, m_kali_1d, np.array(maes, dtype=np.uint8)))
    output_file = fileasli + "_encrypted"

    with open(output_file, "wb") as f:
        m_cipher.tofile(f)

    print("\nEncryption Finished")
    return output_file

# ============================================================
# DECRYPTION
# ============================================================

def dekripsi(fileasli, password2):
    mc = np.fromfile(fileasli, dtype=np.uint8)
    n = int.from_bytes(bytes(mc[:4]), byteorder='big')
    len_hill = int.from_bytes(bytes(mc[4:12]), byteorder='big')

    hill_part = mc[12:12 + len_hill]
    aes_part = mc[12 + len_hill:]

    p, q = n//2, n - n//2
    x0_str = "0." + str(password2) + "1"

    _, balikku = kunci_permutasi(p, q, 'd')

    mc_reshape = hill_part.reshape((n, int(len_hill / n)))
    m_kali = np.dot(balikku.astype(np.int32), mc_reshape.astype(np.int32)) % 256
    m_kali = m_kali.astype(np.uint8)

    maes = aes_decrypt(aes_part.tolist(), x0_str)

    m_kali_1d = m_kali.reshape(len_hill)
    m_decipher = np.concatenate((m_kali_1d, np.array(maes, dtype=np.uint8)))
    output_file = fileasli + "_decrypted"

    with open(output_file, "wb") as f:
        m_decipher.tofile(f)

    print("\nDecryption Finished")
    return output_file

# ============================================================
# MAIN MENU
# ============================================================

print("=" * 80)
print("Hybrid Cryptosystem")
print("Permutation Matrix Hill Cipher + AES-CBC")
print("=" * 80)

print("\nSelect Mode")
print("1 = Encrypt File")
print("2 = Decrypt File")

mode = input("\nYour choice: ")

if mode == "1":
    uploaded = files.upload()
    fileku = list(uploaded.keys())[0]
    password2 = input("\nPassword 2 (digits only): ")
    hasil = enkripsi(fileku, password2)
    files.download(hasil)

elif mode == "2":
    uploaded = files.upload()
    fileku = list(uploaded.keys())[0]
    password2 = input("\nPassword 2: ")
    hasil = dekripsi(fileku, password2)
    files.download(hasil)

else:
    print("\nInvalid option")


In [ ]:
# ============================================================
# GOOGLE COLAB VERSION
# AI-Assisted Hybrid Cryptosystem with Full Performance Metrics
# Permutation Matrix (from Bipartite Graph Kp,q) + AES-CBC
# ============================================================

!pip install pycryptodome psutil

import numpy as np
import os
import time
import hashlib
import tracemalloc
import psutil

from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from google.colab import files

# ============================================================
# ENTROPY CALCULATION
# ============================================================

def hitung_entropy(data):
    hist = np.bincount(data, minlength=256)
    prob = hist / np.sum(hist)
    entropy = -np.sum([p * np.log2(p) for p in prob if p > 0])
    return entropy

# ============================================================
# METRICS REPORTING SYSTEM
# ============================================================

def cetak_laporan_performa(title, elapsed_time, peak_memory_kb, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy):
    throughput_mbps = (orig_size / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0.0
    overhead = proc_size - orig_size

    print("\n" + "=" * 80)
    print(f" METRIKS EVALUASI PERFORMA METODE ({title.upper()})")
    print("=" * 80)
    print(f" [1] Waktu Eksekusi         : {elapsed_time:.4f} detik")
    print(f" [2] Throughput             : {throughput_mbps:.2f} MB/detik")
    print(f" [3] Peak Memory (RAM)      : {peak_memory_kb:.2f} KB ({peak_memory_kb / 1024:.2f} MB)")
    print(f" [4] Penggunaan CPU         : {cpu_usage:.1f} %")
    print(f" [5] Ukuran File Input      : {orig_size} bytes")
    print(f" [6] Ukuran File Output     : {proc_size} bytes (Overhead: {overhead:+d} bytes)")
    print(f" [7] Shannon Entropy (Input): {orig_entropy:.4f} / 8.0")
    print(f" [8] Shannon Entropy (Output): {proc_entropy:.4f} / 8.0")
    print("=" * 80 + "\n")

# ============================================================
# AI MATRIX SIZE SELECTOR
# ============================================================

def pilih_matriks_ai(data):
    filesize = len(data)
    entropy = hitung_entropy(data)

    if filesize < 50000:
        n = 4 if entropy < 5 else 6
    elif filesize < 500000:
        n = 8 if entropy < 6 else 12
    elif filesize < 5000000:
        n = 16 if entropy < 7 else 24
    else:
        n = 32 if entropy < 7.5 else 48

    print(f"AI Selected Matrix Size : {n}")
    return n

# ============================================================
# PERMUTATION MATRIX GENERATOR (from Kp,q)
# ============================================================

def kunci_permutasi(p, q, ed='e'):
    n = p + q
    P = np.eye(n, dtype=np.int32)

    perm = list(range(n))
    if q >= 2:
        perm[p], perm[p+1] = perm[p+1], perm[p]
    if q >= 4:
        perm[p+2], perm[p+3] = perm[p+3], perm[p+2]

    P = P[perm, :]

    if ed == 'e':
        return P, 0
    elif ed == 'd':
        return P, P.T   # invers = transpose

# ============================================================
# AES ENCRYPTION & DECRYPTION
# ============================================================

def aes_encrypt(pesan, kunci):
    data = bytes(pesan)
    key = hashlib.sha256(kunci.encode()).digest()
    cipher = AES.new(key, AES.MODE_CBC)
    iv = cipher.iv
    padded_data = pad(data, AES.block_size)
    ciphertext = cipher.encrypt(padded_data)
    return list(iv + ciphertext)

def aes_decrypt(data, kunci):
    key = hashlib.sha256(kunci.encode()).digest()
    data = bytes(data)
    iv = data[:AES.block_size]
    ciphertext = data[AES.block_size:]
    cipher = AES.new(key, AES.MODE_CBC, iv=iv)
    decrypted_padded = cipher.decrypt(ciphertext)
    decrypted = unpad(decrypted_padded, AES.block_size)
    return list(decrypted)

# ============================================================
# ENCRYPTION WITH PERFORMANCE MONITORING
# ============================================================

def enkripsi(fileasli, password2):
    # Persiapan Pemantauan
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    process.cpu_percent(interval=None) # Reset penghitung CPU
    start_time = time.time()

    # Alur Enkripsi
    mp = np.fromfile(fileasli, dtype=np.uint8)
    orig_size = len(mp)
    orig_entropy = hitung_entropy(mp)

    n = pilih_matriks_ai(mp)
    p, q = n//2, n - n//2
    x0_str = "0." + str(password2) + "1"

    len_shift = mp.size % n
    len_hill = mp.size - len_shift

    kunciku, _ = kunci_permutasi(p, q, 'e')

    mp_reshape = mp[:len_hill].reshape((n, int(len_hill / n)))
    m_kali = np.dot(kunciku.astype(np.int32), mp_reshape.astype(np.int32)) % 256
    m_kali = m_kali.astype(np.uint8)

    maes = aes_encrypt(mp[len_hill:], x0_str)

    m_kali_1d = m_kali.reshape(len_hill)
    header_n = np.array(list(n.to_bytes(4, byteorder='big')), dtype=np.uint8)
    header_len = np.array(list(len_hill.to_bytes(8, byteorder='big')), dtype=np.uint8)

    m_cipher = np.concatenate((header_n, header_len, m_kali_1d, np.array(maes, dtype=np.uint8)))
    output_file = fileasli + "_encrypted"

    with open(output_file, "wb") as f:
        m_cipher.tofile(f)

    # Akhir Pemantauan Performa
    elapsed_time = time.time() - start_time
    proc_entropy = hitung_entropy(m_cipher)
    proc_size = len(m_cipher)
    cpu_usage = process.cpu_percent(interval=None)
    _, peak_mem_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cetak_laporan_performa("ENKRIPSI", elapsed_time, peak_mem_bytes / 1024, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy)
    return output_file

# ============================================================
# DECRYPTION WITH PERFORMANCE MONITORING
# ============================================================

def dekripsi(fileasli, password2):
    # Persiapan Pemantauan
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    process.cpu_percent(interval=None) # Reset penghitung CPU
    start_time = time.time()

    # Alur Dekripsi
    mc = np.fromfile(fileasli, dtype=np.uint8)
    orig_size = len(mc)
    orig_entropy = hitung_entropy(mc)

    n = int.from_bytes(bytes(mc[:4]), byteorder='big')
    len_hill = int.from_bytes(bytes(mc[4:12]), byteorder='big')

    hill_part = mc[12:12 + len_hill]
    aes_part = mc[12 + len_hill:]

    p, q = n//2, n - n//2
    x0_str = "0." + str(password2) + "1"

    _, balikku = kunci_permutasi(p, q, 'd')

    mc_reshape = hill_part.reshape((n, int(len_hill / n)))
    m_kali = np.dot(balikku.astype(np.int32), mc_reshape.astype(np.int32)) % 256
    m_kali = m_kali.astype(np.uint8)

    maes = aes_decrypt(aes_part.tolist(), x0_str)

    m_kali_1d = m_kali.reshape(len_hill)
    m_decipher = np.concatenate((m_kali_1d, np.array(maes, dtype=np.uint8)))
    output_file = fileasli + "_decrypted"

    with open(output_file, "wb") as f:
        m_decipher.tofile(f)

    # Akhir Pemantauan Performa
    elapsed_time = time.time() - start_time
    proc_entropy = hitung_entropy(m_decipher)
    proc_size = len(m_decipher)
    cpu_usage = process.cpu_percent(interval=None)
    _, peak_mem_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cetak_laporan_performa("DEKRIPSI", elapsed_time, peak_mem_bytes / 1024, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy)
    return output_file

# ============================================================
# MAIN MENU
# ============================================================

print("=" * 80)
print("Hybrid Cryptosystem")
print("Permutation Matrix Hill Cipher + AES-CBC (Evaluasi Performa Evaluated)")
print("=" * 80)

print("\nSelect Mode")
print("1 = Encrypt File")
print("2 = Decrypt File")

mode = input("\nYour choice: ")

if mode == "1":
    uploaded = files.upload()
    fileku = list(uploaded.keys())[0]
    password2 = input("\nPassword 2 (digits only): ")
    hasil = enkripsi(fileku, password2)
    files.download(hasil)

elif mode == "2":
    uploaded = files.upload()
    fileku = list(uploaded.keys())[0]
    password2 = input("\nPassword 2: ")
    hasil = dekripsi(fileku, password2)
    files.download(hasil)

else:
    print("\nInvalid option")

In [ ]:
# ============================================================
# GOOGLE COLAB VERSION - HYBRID CRYPTOSYSTEM
# Permutation Matrix (from Bipartite Graph Kp,q) + AES-CBC
# Extended Output & Step-by-Step Visualization
# ============================================================

!pip install pycryptodome psutil

import numpy as np
import os
import time
import hashlib
import tracemalloc
import psutil

from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from google.colab import files

# ============================================================
# HELPER FOR MATRIX / ARRAY PRINTING
# ============================================================

def print_array_preview(name, arr, max_items=10):
    """Menampilkan ringkasan array/matriks agar tampilan konsol tetap rapi."""
    arr_np = np.array(arr)
    print(f"\n--- [VISUALISASI] {name} ---")
    print(f"  • Shape / Ukuran : {arr_np.shape}")
    print(f"  • Data Type      : {arr_np.dtype}")
    if arr_np.ndim == 1:
        if len(arr_np) <= max_items:
            print(f"  • Isi Data       : {arr_np.tolist()}")
        else:
            sample = arr_np[:max_items].tolist()
            print(f"  • Sample Data    : {sample} ... (total {len(arr_np)} elemen)")
    elif arr_np.ndim == 2:
        rows, cols = arr_np.shape
        print(f"  • Baris x Kolom  : {rows} x {cols}")
        if rows <= max_items and cols <= max_items:
            print(f"  • Isi Matriks    :\n{arr_np}")
        else:
            sub_r = min(rows, 6)
            sub_c = min(cols, 6)
            print(f"  • Sample Matriks ({sub_r}x{sub_c}) :\n{arr_np[:sub_r, :sub_c]}")

# ============================================================
# ENTROPY CALCULATION
# ============================================================

def hitung_entropy(data):
    hist = np.bincount(data, minlength=256)
    prob = hist / np.sum(hist)
    entropy = -np.sum([p * np.log2(p) for p in prob if p > 0])
    return entropy

# ============================================================
# METRICS REPORTING SYSTEM
# ============================================================

def cetak_laporan_performa(title, elapsed_time, peak_memory_kb, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy):
    throughput_mbps = (orig_size / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0.0
    overhead = proc_size - orig_size

    print("\n" + "=" * 80)
    print(f" METRIKS EVALUASI PERFORMA METODE ({title.upper()})")
    print("=" * 80)
    print(f" [1] Waktu Eksekusi         : {elapsed_time:.4f} detik")
    print(f" [2] Throughput             : {throughput_mbps:.2f} MB/detik")
    print(f" [3] Peak Memory (RAM)      : {peak_memory_kb:.2f} KB ({peak_memory_kb / 1024:.2f} MB)")
    print(f" [4] Penggunaan CPU         : {cpu_usage:.1f} %")
    print(f" [5] Ukuran File Input      : {orig_size} bytes")
    print(f" [6] Ukuran File Output     : {proc_size} bytes (Overhead: {overhead:+d} bytes)")
    print(f" [7] Shannon Entropy (Input): {orig_entropy:.4f} / 8.0")
    print(f" [8] Shannon Entropy (Output): {proc_entropy:.4f} / 8.0")
    print("=" * 80 + "\n")

# ============================================================
# AI MATRIX SIZE SELECTOR
# ============================================================

def pilih_matriks_ai(data):
    filesize = len(data)
    entropy = hitung_entropy(data)

    if filesize < 50000:
        n = 4 if entropy < 5 else 6
    elif filesize < 500000:
        n = 8 if entropy < 6 else 12
    elif filesize < 5000000:
        n = 16 if entropy < 7 else 24
    else:
        n = 32 if entropy < 7.5 else 48

    print(f"\n[AI SELECTOR]")
    print(f"  • File Size         : {filesize} bytes")
    print(f"  • Shannon Entropy   : {entropy:.4f}")
    print(f"  • AI Selected Matrix Size (n) : {n}")
    return n

# ============================================================
# PERMUTATION MATRIX GENERATOR (from Kp,q)
# ============================================================

def kunci_permutasi(p, q, ed='e'):
    n = p + q
    P_identity = np.eye(n, dtype=np.int32)

    perm = list(range(n))
    if q >= 2:
        perm[p], perm[p+1] = perm[p+1], perm[p]
    if q >= 4:
        perm[p+2], perm[p+3] = perm[p+3], perm[p+2]

    P_permuted = P_identity[perm, :]

    if ed == 'e':
        return P_permuted, P_identity
    elif ed == 'd':
        return P_permuted, P_permuted.T   # invers = transpose

# ============================================================
# AES ENCRYPTION & DECRYPTION
# ============================================================

def aes_encrypt(pesan, kunci):
    data = bytes(pesan)
    key = hashlib.sha256(kunci.encode()).digest()
    cipher = AES.new(key, AES.MODE_CBC)
    iv = cipher.iv
    padded_data = pad(data, AES.block_size)
    ciphertext = cipher.encrypt(padded_data)
    return list(iv + ciphertext)

def aes_decrypt(data, kunci):
    key = hashlib.sha256(kunci.encode()).digest()
    data = bytes(data)
    iv = data[:AES.block_size]
    ciphertext = data[AES.block_size:]
    cipher = AES.new(key, AES.MODE_CBC, iv=iv)
    decrypted_padded = cipher.decrypt(ciphertext)
    decrypted = unpad(decrypted_padded, AES.block_size)
    return list(decrypted)

# ============================================================
# ENCRYPTION PROCESS
# ============================================================

def enkripsi(fileasli, password2):
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    process.cpu_percent(interval=None)
    start_time = time.time()

    print("\n" + "=" * 80)
    print(" 1. ALUR ENKRIPSI")
    print("=" * 80)

    # ------------------------------------------------------------
    # A. Analisis File & Pemilihan Ukuran Matriks (AI Selector)
    # ------------------------------------------------------------
    print("\n>>> [A] Analisis File & Pemilihan Ukuran Matriks (AI Selector)")
    mp = np.fromfile(fileasli, dtype=np.uint8)
    orig_size = len(mp)
    orig_entropy = hitung_entropy(mp)

    print_array_preview("Data Byte File Digital (Input)", mp)
    print(f"  • Nilai Shannon Entropy Input: {orig_entropy:.4f} / 8.0")

    n = pilih_matriks_ai(mp)

    # ------------------------------------------------------------
    # B. Generasi Kunci Permutasi Graf Bipartit (Kp,q)
    # ------------------------------------------------------------
    print("\n>>> [B] Generasi Kunci Permutasi Graf Bipartit (Kp,q)")
    p, q = n // 2, n - (n // 2)
    print(f"  • Nilai n = {n} dibagi menjadi partisi: p = {p} (floor(n/2)) dan q = {q} (n - p)")

    kunciku, P_identity = kunci_permutasi(p, q, 'e')
    print_array_preview("Matriks Identitas Awal (P)", P_identity)
    print_array_preview("Matriks Permutasi P Terbentuk (Setelah Penukaran Baris Partisi V)", kunciku)

    # ------------------------------------------------------------
    # C. Pembagian Data File
    # ------------------------------------------------------------
    print("\n>>> [C] Pembagian Data File")
    len_shift = mp.size % n
    len_hill = mp.size - len_shift

    print(f"  • Total Ukuran File : {mp.size} bytes")
    print(f"  • Sisa Pembagian n (len_shift untuk AES) : {len_shift} bytes")
    print(f"  • Ukuran Bagian Utama (len_hill untuk Hill Cipher) : {len_hill} bytes")

    mp_hill_part = mp[:len_hill]
    mp_aes_part = mp[len_hill:]

    mp_reshape = mp_hill_part.reshape((n, int(len_hill / n)))
    print_array_preview("Matriks Data Utama (Reshaped n x (len_hill/n))", mp_reshape)
    print_array_preview("Sisa Data (len_shift) yang dikirim ke AES", mp_aes_part)

    # ------------------------------------------------------------
    # D. Proses Enkripsi Hibrida
    # ------------------------------------------------------------
    print("\n>>> [D] Proses Enkripsi Hibrida")
    # 1. Permutation Matrix (Hill Cipher)
    m_kali = np.dot(kunciku.astype(np.int32), mp_reshape.astype(np.int32)) % 256
    m_kali = m_kali.astype(np.uint8)
    print_array_preview("Hasil Perkalian Matriks Permutasi (Modulo 256)", m_kali)

    # 2. AES-CBC Encryption
    x0_str = "0." + str(password2) + "1"
    maes = aes_encrypt(mp_aes_part, x0_str)
    print_array_preview("Hasil Enkripsi AES-CBC pada Sisa Data (IV + Ciphertext)", maes)

    # ------------------------------------------------------------
    # E. Penyusunan Output & Header
    # ------------------------------------------------------------
    print("\n>>> [E] Penyusunan Output & Header")
    m_kali_1d = m_kali.reshape(len_hill)

    header_n = np.array(list(n.to_bytes(4, byteorder='big')), dtype=np.uint8)
    header_len = np.array(list(len_hill.to_bytes(8, byteorder='big')), dtype=np.uint8)

    print_array_preview("Header n (4 Byte)", header_n)
    print_array_preview("Header len_hill (8 Byte)", header_len)

    m_cipher = np.concatenate((header_n, header_len, m_kali_1d, np.array(maes, dtype=np.uint8)))
    print_array_preview("File Terenkripsi Final ([Header n] + [Header len_hill] + [Hill Cipher] + [AES])", m_cipher)

    output_file = fileasli + "_encrypted"
    with open(output_file, "wb") as f:
        m_cipher.tofile(f)

    print(f"\n[Selesai] File terenkripsi disimpan sebagai: {output_file}")

    elapsed_time = time.time() - start_time
    proc_entropy = hitung_entropy(m_cipher)
    proc_size = len(m_cipher)
    cpu_usage = process.cpu_percent(interval=None)
    _, peak_mem_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cetak_laporan_performa("ENKRIPSI", elapsed_time, peak_mem_bytes / 1024, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy)
    return output_file

# ============================================================
# DECRYPTION PROCESS
# ============================================================

def dekripsi(fileasli, password2):
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    process.cpu_percent(interval=None)
    start_time = time.time()

    print("\n" + "=" * 80)
    print(" 2. ALUR DEKRIPSI")
    print("=" * 80)

    mc = np.fromfile(fileasli, dtype=np.uint8)
    orig_size = len(mc)
    orig_entropy = hitung_entropy(mc)

    # ------------------------------------------------------------
    # A. Pembacaan Header & Pemisahan Data
    # ------------------------------------------------------------
    print("\n>>> [A] Pembacaan Header & Pemisahan Data")
    n = int.from_bytes(bytes(mc[:4]), byteorder='big')
    len_hill = int.from_bytes(bytes(mc[4:12]), byteorder='big')

    print(f"  • Membaca Header 4-byte pertama  -> Ukuran Matriks n = {n}")
    print(f"  • Membaca Header 8-byte berikutnya -> Panjang Data Hill Cipher (len_hill) = {len_hill} bytes")

    hill_part = mc[12:12 + len_hill]
    aes_part = mc[12 + len_hill:]

    print_array_preview("Data Bagian Hill Cipher (hill_part)", hill_part)
    print_array_preview("Data Bagian AES (aes_part)", aes_part)

    # ------------------------------------------------------------
    # B. Proses Dekripsi Hibrida
    # ------------------------------------------------------------
    print("\n>>> [B] Proses Dekripsi Hibrida")
    p, q = n // 2, n - (n // 2)
    x0_str = "0." + str(password2) + "1"

    # 1. Inverse Permutation
    P_permuted, P_transpose = kunci_permutasi(p, q, 'd')
    print_array_preview("Matriks Permutasi P (Diperoleh Kembali)", P_permuted)
    print_array_preview("Matriks Transpose P^T (Sebagai Invers Matriks)", P_transpose)

    mc_reshape = hill_part.reshape((n, int(len_hill / n)))
    m_kali = np.dot(P_transpose.astype(np.int32), mc_reshape.astype(np.int32)) % 256
    m_kali = m_kali.astype(np.uint8)
    print_array_preview("Matriks Hasil Perkalian P^T dengan hill_part (Modulo 256)", m_kali)

    # 2. AES-CBC Decryption
    maes = aes_decrypt(aes_part.tolist(), x0_str)
    print_array_preview("Hasil Dekripsi AES-CBC Sisa Data", maes)

    # ------------------------------------------------------------
    # C. Penggabungan File
    # ------------------------------------------------------------
    print("\n>>> [C] Penggabungan File")
    m_kali_1d = m_kali.reshape(len_hill)
    m_decipher = np.concatenate((m_kali_1d, np.array(maes, dtype=np.uint8)))

    print_array_preview("Hasil Penggabungan Akhir (File Asli Terpulihkan)", m_decipher)

    output_file = fileasli + "_decrypted"
    with open(output_file, "wb") as f:
        m_decipher.tofile(f)

    print(f"\n[Selesai] File terdekripsi disimpan sebagai: {output_file}")

    elapsed_time = time.time() - start_time
    proc_entropy = hitung_entropy(m_decipher)
    proc_size = len(m_decipher)
    cpu_usage = process.cpu_percent(interval=None)
    _, peak_mem_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cetak_laporan_performa("DEKRIPSI", elapsed_time, peak_mem_bytes / 1024, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy)
    return output_file

# ============================================================
# MAIN MENU
# ============================================================

print("=" * 80)
print("Hybrid Cryptosystem")
print("Permutation Matrix Hill Cipher + AES-CBC (Detailed Execution Flow)")
print("=" * 80)

print("\nSelect Mode")
print("1 = Encrypt File")
print("2 = Decrypt File")

mode = input("\nYour choice: ")

if mode == "1":
    uploaded = files.upload()
    fileku = list(uploaded.keys())[0]
    password2 = input("\nPassword 2 (digits only): ")
    hasil = enkripsi(fileku, password2)
    files.download(hasil)

elif mode == "2":
    uploaded = files.upload()
    fileku = list(uploaded.keys())[0]
    password2 = input("\nPassword 2: ")
    hasil = dekripsi(fileku, password2)
    files.download(hasil)

else:
    print("\nInvalid option")

Hybrid Cryptosystem
Permutation Matrix Hill Cipher + AES-CBC (Detailed Execution Flow)

Select Mode
1 = Encrypt File
2 = Decrypt File

Your choice: 1


Saving COMNET-S-26-08200.pdf to COMNET-S-26-08200 (2).pdf

Password 2 (digits only): 222

 1. ALUR ENKRIPSI

>>> [A] Analisis File & Pemilihan Ukuran Matriks (AI Selector)

--- [VISUALISASI] Data Byte File Digital (Input) ---
  • Shape / Ukuran : (2006753,)
  • Data Type      : uint8
  • Sample Data    : [37, 80, 68, 70, 45, 49, 46, 52, 10, 37] ... (total 2006753 elemen)
  • Nilai Shannon Entropy Input: 7.9861 / 8.0

[AI SELECTOR]
  • File Size         : 2006753 bytes
  • Shannon Entropy   : 7.9861
  • AI Selected Matrix Size (n) : 24

>>> [B] Generasi Kunci Permutasi Graf Bipartit (Kp,q)
  • Nilai n = 24 dibagi menjadi partisi: p = 12 (floor(n/2)) dan q = 12 (n - p)

--- [VISUALISASI] Matriks Identitas Awal (P) ---
  • Shape / Ukuran : (24, 24)
  • Data Type      : int32
  • Baris x Kolom  : 24 x 24
  • Sample Matriks (6x6) :
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]]

--- [VISUALISASI] Matriks Permutasi P Terbentuk (Setelah Penukaran Ba

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>